In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, average_precision_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.nn import BCEWithLogitsLoss
from datasets import Dataset
import os
import gc


torch.cuda.empty_cache()
gc.collect()


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model_name = 'ai-forever/ruRoberta-large'
batch_size = 32
main_seed = 42
epochs = 20


import random
random.seed(main_seed)
np.random.seed(main_seed)
torch.manual_seed(main_seed)
torch.cuda.manual_seed_all(main_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=1).to(device)


window_context = 3

train_dataset = pd.DataFrame({'text': {}, 'label' : {}})
test_dataset = pd.DataFrame({'text': {}, 'label' : {}})

train_files, test_files = train_test_split(os.listdir('datasets'), test_size=0.25, random_state=42)


for file in train_files:
    temp_dataset = pd.read_csv('datasets/'+file)

    if window_context > 0:
        texts = []
        new_labels = []
        for i in range(window_context, len(temp_dataset) - window_context):
            text = ''
                
            for j in range(i - window_context, i + window_context + 1):
                text += temp_dataset.iloc[j]['text'] + ' '

            texts.append(text.strip())
            new_labels.append(temp_dataset.iloc[i]['label'])

        train_dataset = pd.concat([train_dataset, pd.DataFrame({'text': texts, 'label': new_labels})], ignore_index=True)
    else:
        train_dataset = pd.concat([train_dataset, temp_dataset], ignore_index=True)


for file in test_files:
    temp_dataset = pd.read_csv('datasets/'+file)

    if window_context > 0:
        texts = []
        new_labels = []
        for i in range(window_context, len(temp_dataset) - window_context):
            text = ''
                
            for j in range(i - window_context, i + window_context + 1):
                text += temp_dataset.iloc[j]['text'] + ' '

            texts.append(text.strip())
            new_labels.append(temp_dataset.iloc[i]['label'])

        test_dataset = pd.concat([test_dataset, pd.DataFrame({'text': texts, 'label': new_labels})], ignore_index=True)
    else:
        test_dataset = pd.concat([test_dataset, temp_dataset], ignore_index=True)




specific_dataset = pd.DataFrame({'text': {}, 'label' : {}})

for file in os.listdir('datasets_twitch'):
    temp_dataset = pd.read_csv('datasets_twitch/'+file)

    if window_context > 0:
        texts = []
        new_labels = []
        for i in range(window_context, len(temp_dataset) - window_context):
            text = ''
                
            for j in range(i - window_context, i + window_context + 1):
                text += temp_dataset.iloc[j]['text'] + ' '

            texts.append(text.strip())
            new_labels.append(temp_dataset.iloc[i]['label'])

        specific_dataset = pd.concat([specific_dataset, pd.DataFrame({'text': texts, 'label': new_labels})], ignore_index=True)
    else:
        specific_dataset = pd.concat([specific_dataset, temp_dataset], ignore_index=True)


for file in os.listdir('datasets_synthetic'):
    temp_dataset = pd.read_csv('datasets_synthetic/'+file)
    specific_dataset = pd.concat([specific_dataset, temp_dataset], ignore_index=True)

specific_dataset = specific_dataset.drop_duplicates(subset=["text"])
specific_dataset = specific_dataset.dropna(subset=["text"])
train_dataset = train_dataset.drop_duplicates(subset=["text"])
test_dataset = test_dataset.drop_duplicates(subset=["text"])



train_dataset = Dataset.from_pandas(pd.concat([train_dataset, specific_dataset], ignore_index=True))
test_dataset = Dataset.from_pandas(test_dataset)


def preprocess(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=512)

train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])


train_labels = np.array(train_dataset['label'])
neg_count = np.sum(train_labels == 0)
pos_count = np.sum(train_labels == 1)
print(f"\nВеса классов: neg={neg_count}, pos={pos_count}")

if pos_count > 0:
    pos_weight = torch.tensor([neg_count / pos_count]).to(device)
    print(f"pos_weight: {pos_weight.item()}")
else:
    pos_weight = torch.tensor([1.0]).to(device)


class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = BCEWithLogitsLoss(pos_weight=pos_weight)

        labels = labels.float().view(-1, 1)
        logits = logits.view(-1, 1)
        
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy().flatten()
    labels = labels.flatten()
    preds = (probs >= 0.5).astype(int)
    
    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted', zero_division=0)
    precision = precision_score(labels, preds, average='binary', zero_division=0)
    recall = recall_score(labels, preds, average='binary', zero_division=0)
    roc_auc = roc_auc_score(labels, probs)

    cm = confusion_matrix(labels, preds)
    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"]
    )
    print("\nМатрица ошибок (Confusion Matrix):")
    print(cm_df)
    return {
        'accuracy': accuracy,'f1': f1,
        'precision': precision,'recall': recall,  'roc_auc': roc_auc
    }

training_args = TrainingArguments(
    output_dir="./roberta3",
    num_train_epochs=epochs,
    eval_steps=1000,
    save_steps=1000,
    logging_steps=1000,

    # eval_strategy="epoch",
    # save_strategy="epoch",

    eval_strategy='steps',
    save_strategy='steps',

    learning_rate=1e-6,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.05,
    warmup_steps=500,
    logging_dir='./logs',
    load_best_model_at_end=True,
    save_total_limit=20,
    report_to="tensorboard",
    metric_for_best_model="f1",
    fp16=True,
    greater_is_better=True,
    do_train=True,
    do_eval=True,
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

trainer.save_model("./roberta3")
tokenizer.save_pretrained("./roberta3")